### Mapping human lymph node cell types to 10X Visium with Cell2location

This tutorial shows how to use **cell2location** method for spatially resolving fine-grained cell types by integrating 10X Visium data with scRNA-seq reference of cell types. Cell2location is a principled Bayesian model that estimates which combination of cell types in which cell abundance could have given the mRNA counts in the spatial data, while modelling technical effects (platform/technology effect, contaminating RNA, unexplained variance).

<div class="alert alert-info">
<b>Note!</b>

Cell2location is an independent package, but is powered by scvi-tools. If you have questions about cell2location, Visium data or scvi-tools please visit https://discourse.scverse.org/c/ecosytem/cell2location/42, https://discourse.scverse.org/c/general/visium/32 or https://discourse.scverse.org/c/help/scvi-tools/7 correspondingly.

</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BayraktarLab/cell2location/blob/master/docs/notebooks/cell2location_tutorial.ipynb)

In this tutorial, we analyse a publicly available Visium dataset of the human lymph node from 10X Genomics, and spatially map a comprehensive atlas of 34 reference cell types derived by integration of scRNA-seq datasets from human secondary lymphoid organs.

- Cell2location provides high sensitivity and resolution by borrowing statistical strength across locations. This is achieved by modelling similarity of location patterns between cell types using a hierarchical factorisation of cell abundance into tissue zones as a prior (see paper methods).

- Using our statistical method based on Negative Binomial regression to robustly combine scRNA-seq reference data across technologies and batches results in improved spatial mapping accuracy. Given cell type annotation for each cell, the corresponding reference cell type signatures $g_{f,g}$, which represent the average mRNA count of each gene $g$ in each cell type $f$, can be estimated from sc/snRNA-seq data using either 1) NB regression or 2) a hard-coded computation of per-cluster average mRNA counts for individual genes.  We generally recommend using NB regression. This notebook shows use a dataset composed on multiple batches and technologies.When the batch effects are small, a faster hard-coded method of computing per cluster averages provides similarly high accuracy. We also recommend the hard-coded method for non-UMI technologies such as Smart-Seq 2.

- Cell2location needs untransformed unnormalised spatial mRNA counts as input.

- You also need to provide cell2location with the expected average cell abundance per location which is used as a prior to guide estimation of absolute cell abundance. This value depends on the tissue and can be estimated by counting nuclei for a few locations in the paired histology image but can be approximate (see [paper methods for more guidance](https://github.com/BayraktarLab/cell2location/blob/master/docs/images/Note_on_selecting_hyperparameters.pdf)).


## Workflow diagram

see: https://cell2location.readthedocs.io/en/latest/notebooks/cell2location_tutorial.html

## Contents
* [Loading packages](#Loading-packages)
* [Loading Visium and single cell data data](#Loading-Visium-and-scRNA-seq-reference-data)
1. [Estimating cell type signatures (NB regression)](#Estimation-of-reference-cell-type-signatures-(NB-regression))
2. [Cell2location: spatial mapping](#Cell2location:-spatial-mapping)
3. [Visualising cell abundance in spatial coordinates](#Visualising-cell-abundance-in-spatial-coordinates)
4. [Downstream analysis](#Downstream-analysis)
* [Leiden clustering of cell abundance](#Identifying-discrete-tissue-regions-by-Leiden-clustering)
* [Identifying cellular compartments / tissue zones using non-negative matrix factorisation (NMF)](#Identifying-cellular-compartments-/-tissue-zones-using-matrix-factorisation-(NMF))
* [Estimate cell-type specific expression of every gene in the spatial data (needed for NCEM)](#Estimate-cell-type-specific-expression-of-every-gene-in-the-spatial-data-(needed-for-NCEM))
5. [Advanced use](#Advanced-use)
* [Working with the posterior distribution and computing arbitrary quantiles](#Working-with-the-posterior-distribution-and-computing-arbitrary-quantiles)

### Slide-seq 

Slide-seq is a high-resolution spatial transcriptomics technology developed by researchers at the Broad Institute of MIT and Harvard (Fei Chen and Evan Macosko). It measures genome-wide gene expression within tissue sections while retaining the precise spatial location of cells. [1](https://www.science.org/doi/10.1126/science.aaw1219), [2](https://www.broadinstitute.org/news/new-tool-uses-rna-sequencing-chart-rich-maps-cellular-and-tissue-function), [3](https://commonfund.nih.gov/earlyindependence/highlights/slide-seq-novel-method-map-tissue-cells), [4](https://www.broadinstitute.org/publications/broad508511)

How It Works

* DNA-Barcoded Beads ("Pucks"): A dense monolayer of 10-micron DNA-barcoded beads (comparable to the size of individual cells) is packed onto a surface called a puck. [1](https://pubmed.ncbi.nlm.nih.gov/30923225/), [2](https://www.nature.com/articles/s41587-020-0739-1), [3](https://www.science.org/doi/10.1126/science.aaw1219)
* RNA Transfer: Fresh-frozen tissue sections are placed directly onto the puck, allowing cellular RNA to transfer onto the underlying beads. [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC11254742/), [2](https://pubmed.ncbi.nlm.nih.gov/30923225/)
* Sequencing & Mapping: Standard high-throughput RNA sequencing is performed. By matching the RNA sequences back to the known spatial coordinates of the bead barcodes, researchers map gene expression across the tissue without needing a microscope for primary localization. [1](https://www.youtube.com/watch?v=mQXr7ZsojpE&t=213), [2](https://www.broadinstitute.org/news/new-tool-uses-rna-sequencing-chart-rich-maps-cellular-and-tissue-function)

Key Features & Versions

* Original Slide-seq (2019): Achieved 10-micron spatial resolution and enabled 3D tissue volume reconstruction (such as mapping the mouse hippocampus and cerebellum). [1](https://pmc.ncbi.nlm.nih.gov/articles/PMC8606189/), [2](https://www.science.org/doi/10.1126/science.aaw1219), [3](https://www.broadinstitute.org/publications/broad508511)
* Slide-seqV2 (2020): Improved bead synthesis, library generation, and array indexing to capture roughly 50% of single-cell RNA-seq efficiency—a ~10-fold increase in transcript capture over the original method. [1](https://www.nature.com/articles/s41587-020-0739-1), [2](https://www.youtube.com/watch?v=mQXr7ZsojpE&t=213)
* Computational Pipeline: Processed using open-source, cloud-optimized workflows like the Broad Institute WARP Slide-seq Pipeline available on platforms like Terra. Commercialized assays are also known under tools like Curio Seeker. [1](https://broadinstitute.github.io/warp/docs/Pipelines/SlideSeq_Pipeline/README)



## Loading packages <a class="anchor" id="Loading-packages"></a>

```Bash
uv add scvi-colab
uv add cell2location
```

In [ ]:
import os, sys
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path

import scipy.sparse as sp
import squidpy as sq
import cell2location

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.colors import LinearSegmentedColormap

rcParams['pdf.fonttype']= 42 # enables correct plotting of text for PDFs

import warnings

ROOT_SRC = Path("/home/flavio/uv/squidpy/src")
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import *
from libs.cell2location_lib import *          # clip_for_plot, create_gene_age_tissue_groups, plot_panel_umap_by_tissue_age, ...

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")


First, let's define where we save the results of our analysis:

In [ ]:
root0 = Path('/home/flavio/uv/squidpy')
root_results = root0 / 'results'
root_figure = root0 / 'figures'

os.listdir(root_figure)[:3]

In [ ]:
results_folder =  root_results / 'lymph_nodes_analysis/'

# create paths and names to results folders for reference regression and cell2location models
ref_run_name = Path(f'{results_folder}/reference_signatures')
run_name = Path(f'{results_folder}/cell2location_map')


### Loading Visium and scRNA-seq reference data <a class="anchor" id="Loading-Visium-and-scRNA-seq-reference-data"></a>

First let's read spatial Visium data from 10X Space Ranger output. Here we use lymph node data generated by 10X and presented in [Kleshchevnikov et al (section 4, Fig 4)](https://www.biorxiv.org/content/10.1101/2020.11.15.378125v1). This dataset can be conveniently downloaded and imported using scanpy. See [this tutorial](https://cell2location.readthedocs.io/en/latest/notebooks/cell2location_short_demo.html) for a more extensive and practical example of data loading (multiple visium samples).

In [ ]:
import importlib

importlib.metadata.version('scanpy')

In [ ]:
[x for x in sc.datasets.__dict__ if x.startswith('visium')]

In [ ]:
# include_hires_tiff=True if you also want the full-resolution image

adata_vis = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata_vis.obs['sample']= list(adata_vis.uns['spatial'].keys())[0]

In [ ]:
adata_vis.uns['spatial'].keys()

In [ ]:
adata_vis.obs.head(3)

In [ ]:
adata_vis.var_names[:5]

In [ ]:
adata_vis.var

<div class="alert alert-info">
<b>Note!</b>
    
Here we rename genes to ENSEMBL ID for correct matching between single cell and spatial data - so you can ignore the scanpy suggestion to call `.var_names_make_unique`.
</div>

In [ ]:
adata_vis.var['symbol']= adata_vis.var_names
adata_vis.var.head(6)

In [ ]:
adata_vis.var.set_index('gene_ids', drop=True, inplace=True)
adata_vis.var.head(6)

You can still plot gene expression by name using standard scanpy functions as follows:
**``python
sc.pl.spatial(color='PTPRC', gene_symbols='symbol', ...)
```

<div class="alert alert-info">
<b>Note!</b>
    
Mitochondia-encoded genes (gene names start with prefix mt- or MT-) are irrelevant for spatial mapping because their expression represents technical artifacts in the single cell and nucleus data rather than biological abundance of mitochondria. Yet these genes compose 15-40% of mRNA in each location. Hence, to avoid mapping artifacts we strongly recommend removing mitochondrial genes.
</div>

In [ ]:
"CD109" in list(adata_vis.var.symbol)

In [ ]:
genes = [x for x in adata_vis.var.symbol if x.startswith("CD")]
genes.sort()
np.array(genes)[:20]

### B-cell geneS

Pan B-cell gene and protein markers used to identify B lymphocytes include CD19, MS4A1 (which encodes CD20), CD79A, CD79B, PAX5, and CD22. [1](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/), [2](https://www.panglaodb.se/markers.html?cell_type='B%20cells'), [3](https://www.rpubs.com/daniloceschin/Markers_BCells)

Key B-Cell Gene and Surface Markers

* Core Pan B-Cell Markers:
  * CD19: Expressed from the early pro-B stage through mature B cells; acts as a coreceptor for B-cell receptor (BCR) signaling.
  * MS4A1 (CD20): Encodes the CD20 calcium channel protein; present from the pre-B stage to mature B cells and a primary target for immunotherapies (e.g., rituximab).
  * CD79A / CD79B: Encodes Igα and Igβ, essential signaling components associated with the surface B-cell receptor complex.
  * PAX5: A master transcription factor critical for B-cell commitment and identity.
  * CD22: A sialic acid-binding lectin expressed on mature and developing B cells. [1](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/b-cells-and-plasma-cells/), [2](https://www.antibodies.com/primary-antibodies/cell-markers/immune-cell-markers/b-cell-markers), [3](https://www.abcam.com/en-us/technical-resources/research-areas/marker-guides/b-cell-markers), [4](https://www.youtube.com/watch?v=JUq3K8rtcso&vl=en&t=178), [5](https://www.origene.com/research-areas/b-cell-lineage), [6](https://www.rpubs.com/daniloceschin/Markers_BCells), [7](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/)
* Stage-Specific and Subset Markers:
  * Naïve B Cells: IGHD (IgD), IGHM (IgM), CD27−, IL4R, FCER2 (CD23).
  * Memory B Cells: CD27+, CD24+, and switched or unswitched immunoglobulins (IgG, IgA, IgE).
  * Plasma Cells / Plasmablasts: Downregulate CD19 and MS4A1/CD20 while upregulating CD38, SDC1 (CD138), PRDM1 (Blimp-1), and XBP1. [1](https://www.antibodies.com/primary-antibodies/cell-markers/immune-cell-markers/b-cell-markers), [2](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/), [3](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/b-cells-and-plasma-cells/)



In [ ]:
genes = ['CD19', 'MS4A1',   # MS4A1 = CD20 ('CD20' is not a gene symbol)
         'CD79A', 'CD79B', 'PAX5', 'CD22','CD52', 'CD14', 'CD53', 'CDS2', 'IGHD', 'IGHM', 'CD27']
len(genes)


In [ ]:
# clip_for_plot / _values live in libs/cell2location_lib.py
# (warns about unknown features; non-zero percentile cap for very sparse genes such as CD14)
genes, cols = clip_for_plot(adata_vis, genes, q=99.2, vmin=0)
print(len(genes))

sq.pl.spatial_scatter(
    adata_vis,
    color=cols,
    title=list(genes),            # show original names, not the _plot_ prefix
    img_res_key="hires",
    img_alpha=0.6,
    size=1.3,
    cmap="RdBu_r",
    ncols=4,
    library_id="V1_Human_Lymph_Node",
)

### find mitochondria-encoded (MT) genes

In [ ]:
adata_vis.var['MT_gene']= [gene.startswith('MT-') for gene in adata_vis.var['symbol']]

# remove MT genes for spatial mapping (keeping their counts in the object)
adata_vis.obsm['MT']= adata_vis[:, adata_vis.var['MT_gene'].values].X.toarray()
adata_vis = adata_vis[:, ~adata_vis.var['MT_gene'].values]

Published scRNA-seq datasets of lymph nodes have typically lacked an adequate representation of germinal centre-associated immune cell populations due to age of patient donors. We, therefore, include scRNA-seq datasets spanning lymph nodes, spleen and tonsils in our single-cell reference to ensure that we captured the full diversity of immune cell states likely to exist in the spatial transcriptomic dataset.

Here we download this dataset, import into anndata and change variable names to ENSEMBL gene identifiers.

In [ ]:
root_data = root0 / 'data'
os.listdir(root_data)

In [ ]:
fname_sc = 'lymphoid_organ_sc.h5ad'
filename = root_data / fname_sc

# Read data
if filename.exists():
    adata_ref = sc.read_h5ad(filename)
    print(f"Data read from {filename}")
else:
    adata_ref = sc.read(
        filename,
        backup_url='https://cell2location.cog.sanger.ac.uk/paper/integrated_lymphoid_organ_scrna/RegressionNBV4Torch_57covariates_73260cells_10237genes/sc.h5ad'
    )
    adata_ref.write_h5ad(filename)
    print(f"Data written to {filename}")


### lymphoid_organ_sc.h5ad - reference

`lymphoid_organ_sc.h5ad` is the single-cell reference for the cell2location paper's **lymph node / tonsil analysis**. 

It holds **73,260 cells × 10,237 genes**, 1.66 GB on disk, and contains both raw counts and a fitted reference model. I opened it read-only in backed mode, so the file was not changed.

#### Expression matrix (`X`)
- **Raw UMI counts.** The matrix is sparse (CSR, float32) and every value is a whole number. In the first 2,000 cells the counts run from 1 to 1,276, with about 1,080 detected genes per cell.
- **Genes are already filtered.** 
  - Gene symbols are the index and Ensembl IDs are in `var['GeneID-2']`. 
  - The `n_counts` and `n_genes` columns were computed on the full gene set before filtering, so they are slightly higher than the row sums of `X` (e.g. 3,148 vs 3,122).
- **Cell QC:** 
  - median 3,483 UMIs and 1,657 genes per cell; 
  - minimum 581 UMIs and 698 genes.
  - Mitochondrial fraction (`percent_mito`) is 0–15%, median 4%. 
  - Every cell has `predicted_doublet = False`, so doublets were already removed.

#### Studies and tissues (`obs`)
| Study | Tissue | 3′ (3GEX) | 5′ (5GEX) |
|---|---|---|---|
| King_2020 | Tonsil | 2,002 | 27,703 |
| James_2020 | Mesenteric LN | 10,250 | 4,184 |
| Park_2020 | Spleen / iLN / mLN / tLN | 15,506 | 13,615 |

- **23 samples from 13 donors.** Donor A16 alone contributes 29,121 cells.
- **Ages** run from 0–4 years to 65–70 years.
- **Sex:** 48,545 female and 24,715 male cells.
- **Extra columns:** B-cell receptor fields (`BCELL_CLONE`, `ISOTYPE`, `IGH_MU_FREQ`, `VDJsum`), cell-cycle scores and phase, five Leiden clusterings, and several generations of annotation columns.

#### Cell-type annotations
- **`Subset` (34 types)** is the label set the regression model was trained on. The largest groups are B_mem (13,476), B_naive (8,924), T_CD4+_naive (6,012), B_Cycling (4,765) and T_CD4+_TfH (4,690). It also covers germinal-centre B cells (DZ, LZ, prePB, preGC), T-cell types (TfH, Tfr, Treg, CD8 types), NK/NKT/ILC, myeloid cells (DCs, macrophages, monocytes, mast cells) and stromal cells (Endo, FDC, VSMC). Rare types include ILC/Mast (18), VSMC (40) and DC_CCR7+ (42).
- **`Subset_Broad` (38 types)** is finer, despite its name. For example, it splits the endothelium into ACKR1+, CXCL12+ and lymphatic cells.
- **Likely label swap:** `Subset` has ILC = 617 and Mast = 18, while `Subset_Broad` has Mast = 617 and ILC = 18. I only compared the counts and did not cross-tabulate cell by cell. Check marker genes (e.g. TPSAB1/KIT for mast cells) before relying on either label for these two types.

#### Fitted reference model
This file already contains the trained cell2location reference model, so you don't need to retrain it.
- **`uns['regression_mod']`** is a `RegressionNBV4Torch` model with 57 factors: 34 cell types plus 23 sample effects. Its posterior means are:
  - `gene_factors` (57 × 10,237)
  - `sample_scaling` (23 × 1)
  - `tech_scaling` (2 × 10,237, one row per chemistry, 3′ and 5′)
  - `gene_E` (1 × 10,237)
- **In `var`:** 34 `mean_cov_effect_Subset_*` columns hold each cell type's expression signature. These are the columns to pass to cell2location as the reference. There are also 23 `mean_sample_effectSample_*` columns, plus `n_cells` and `nonz_mean`.

#### Embeddings and metadata
- **Embeddings:** `obsm['X_pca']` (100 components), `obsm['X_umap']`, and a neighbour graph built with 69 neighbours (`obsp` connectivities and distances).
- **Colour palettes** are stored in `uns` for the main categories.
- **Unusable columns:** `GeneName-2` holds numbers instead of gene names, and the `feature_types` / `gene_ids-*` columns are mostly NaN. Use the var index for gene names instead.

To extract the signatures for spatial mapping:
```python
sig = adata.var[[c for c in adata.var if c.startswith('mean_cov_effect_Subset_')]]
sig.columns = sig.columns.str.replace('mean_cov_effect_Subset_', '')
```

This was a description only, so I didn't save any files.

In [ ]:
import anndata as ad
import h5py
import scipy.sparse as sp

filename = root_data / fname_sc
print(os.path.getsize(filename)/1e9,"GB")

a=ad.read_h5ad(filename,backed='r')
print(a)

X=a.X
print(type(X), getattr(X,'dtype',None))

with h5py.File(filename,'r') as f:
    g=f['X']; print(dict(g.attrs) if isinstance(g,h5py.Group) else g.shape)
    for k in f['layers'].keys() if 'layers' in f else []: print('layer',k)

In [ ]:
obs=a.obs

for c in ['Study','Tissue','Method','Donor','Sample','Age','Sex','Subset_Broad','Subset','phase','predicted_doublet']:
    vc=obs[c].value_counts(dropna=False)
    print(f"\n## {c} ({vc.size} levels)"); print(vc.head(40).to_string())

print(obs[['n_counts','n_genes','percent_mito','doublet_score']].describe().round(2).to_string())

# X values sample
sub=a.X[:2000]
d=sub.data

print("\n")

print("X sample: min",d.min(),"max",d.max(),"integer?",np.allclose(d,np.round(d)),"nnz/cell",sub.nnz/2000)
print("rowsum vs n_counts", np.asarray(sub.sum(1)).ravel()[:5], obs.n_counts.values[:5])
print(a.var[['GeneName-2','n_cells','nonz_mean']].head().to_string()); print(a.var.index[:5].tolist())
print(a.uns['regression_mod'].keys() if hasattr(a.uns['regression_mod'],'keys') else a.uns['regression_mod'])
print({k:(a.uns[k] if k!='leiden' else a.uns[k]) for k in ['neighbors']})



### Split by Age-group and Tissues

First I'll check how tissue and age overlap in the scRNA reference, and look in the squidpy project for the `genes` list and the `adata_vis` object.

Note: tissue and age are confounded. Tonsil is the only babies/young tissue, and spleen, iLN and tLN are all adult. Let me find the notebook that defines `genes` and `adata_vis`.

This Visium slide has no tissue/age labels, so I'll instead build the B-cell gene panel on the scRNA reference, coloring UMAPs by each gene split by tissue × age group with the same 99.2-percentile clipping. First, I'll load the figure-style skill.

Since CD20 isn't a valid gene symbol (the gene is MS4A1), I'll plot MS4A1 labelled as "MS4A1 (CD20)" instead. Next I'll build the grid with genes in rows and tissue × age groups in columns, using a shared colour scale per gene.

I found two issues to fix before saving: colliding column headers, and a blank CD14 column caused by its extreme sparsity flattening the 99.2-percentile clip to 0. I'll fix the headers and adjust sparse genes like CD14 to use the 99.2th percentile of non-zero values instead.

Yes, but only on the scRNA reference, not on the Visium slide. `V1_Human_Lymph_Node` is a single lymph node section from one donor, so its spots carry no tissue or age labels. Tissue and age exist only in `lymphoid_organ_sc.h5ad`. So I drew the same gene panel and the same 99.2-percentile clipping on the reference UMAP, with one column per tissue × age group.

### B-cell gene panel on the scRNA reference UMAP, split by tissue and age group

**Age groups.** Your cut-offs map onto the reference's age bins like this:

| Group | Age bins | Cells |
|---|---|---|
| babies | 0–4y | 16,237 |
| young | 5–9y | 13,468 |
| adult | 20–25y | 29,121 |
| old adult | 40–44y | 1,825 |
| elder | 55–60y, 60–64y, 65–70y | 12,609 |

**Tissue and age are almost entirely confounded.** Only 8 of the 25 tissue × age combinations contain cells:
- Tonsil exists only as babies and young.
- Spleen, iLN and tLN are adult only, and all come from the single donor A16.
- Only mLN spans three age groups: adult, old adult (one donor, 1,825 cells) and elder (four donors).

So a difference between columns is just as likely to come from tissue, study (King, Park or James) or donor as from age.

**How the plot differs from your Visium version:**
- **CD20 is replaced by MS4A1.** `CD20` is not a gene symbol, so your `clip_for_plot` drops it without a warning, and it is probably missing from your Visium plot too. I plotted `MS4A1`, labelled "MS4A1 (CD20)".
- **CD14 needed a different cap.** More than 99.2% of cells have zero *CD14*, so the 99.2-percentile cap was 0 and the row came out blank. For that gene I clipped at the 99.2th percentile of the non-zero values instead. The same problem can hit any sparse gene in your Visium helper.
- **Expression is normalised.** The reference holds raw counts from mixed 3′ and 5′ chemistry, so I coloured by log1p(counts per 10k), computed per cell. Each gene has one colour scale shared across all columns, so columns can be compared directly.
- **The colour map is sequential.** I used a light-grey-to-red ramp instead of `RdBu_r`, because expression has no meaningful midpoint for a diverging map to centre on.

If you want the split on the slide itself, one option is to build separate cell2location reference signatures per group and map each onto the Visium spots. Given the confounding, though, the result would mostly reflect tissue and study, not age.

Files are in the output tray:
- `bcell_genes_umap_by_tissue_age.png` — the figure above.
- `bcell_genes_by_tissue_age.csv` — per gene and group: cell count, % of cells expressing, and mean log1p(CP10k).

In [ ]:
ad.obs.columns

In [ ]:
ad = adata_ref.copy()   # raw counts, before filter_genes if possible
age_col = "Age"         # check: [c for c in ad.obs.columns if "age" in c.lower()]

def age_group(b):
    lo = int(str(b).split("-")[0].rstrip("y+"))
    return "babies" if lo < 5 else "young" if lo < 14 else "adult" if lo < 34 else "old adult" if lo < 55 else "elder"

ad.obs["age_group"] = ad.obs[age_col].map(age_group)
ml = ad[ad.obs["Tissue"] == "mLN"].copy()      # adjust column name if needed

sc.pp.normalize_total(ml, target_sum=1e4)
sc.pp.log1p(ml)

genes = ["LEF1", "CCR7", "SELL", "TCF7", "CD27", "CD28", "AICDA"]
genes = [g for g in genes if g in ml.var_names]
X = ml[:, genes].X
X = X.toarray() if hasattr(X, "toarray") else X

df = pd.DataFrame(X, columns=genes, index=ml.obs_names)
df["age_group"] = ml.obs["age_group"].values
order = ["adult", "old adult", "elder"]
print(df.groupby("age_group")[genes].mean().reindex(order).round(3))
print((df[genes] > 0).groupby(df["age_group"]).mean().reindex(order).round(3))

del(ad)
df

ve grouped the panel genes into five cell-type markers:

1. pan-B (CD19/MS4A1/CD79A/B/PAX5/CD22)
2. follicular mantle B (IGHD/IGHM)
3. pan-lymphocyte (CD52/CD53)
4. TIM3+ T, GC TfH, Treg, GC DZ B (CD27)
5. monocytes/macrophages (CD14) 

CDS2 doesn't mark any clear cell type.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib as mpl
from scipy.stats import spearmanr
import scipy.sparse as sps
from libs.matplotlib_figure_style import apply_figure_style
import libs.cell2location_lib as L

apply_figure_style(sizes=(8,7,6))

In [ ]:
# lineage hue families (outer level = hue, inner = shade)
lineage = {
 "GC B":      ["B_GC_DZ","B_GC_LZ","B_GC_prePB","B_Cycling","B_preGC"],
 "naive/mem B":["B_naive","B_activated","B_IFN","B_mem"],
 "plasma":    ["B_plasma"],
 "T":         ["T_CD4+_naive","T_CD4+","T_CD4+_TfH","T_CD4+_TfH_GC","T_TfR","T_Treg","T_TIM3+","T_CD8+_naive","T_CD8+_cytotoxic","T_CD8+_CD161+"],
 "NK/ILC":    ["NK","NKT","ILC"],
 "myeloid":   ["Monocytes","Macrophages_M1","Macrophages_M2","DC_cDC1","DC_cDC2","DC_pDC","DC_CCR7+","Mast"],
 "stromal":   ["FDC","Endo","VSMC"],
}

V   = adata_vis
lib = list(V.uns["spatial"].keys())[0]                       # 'V1_Human_Lymph_Node'
Q   = pd.DataFrame(np.asarray(V.obsm["q05_cell_abundance_w_sf"]),   # spots x 34 cell types
                   index=V.obs_names, columns=list(V.uns["mod"]["factor_names"]))

img = V.uns["spatial"][lib]["images"]["hires"]
sf  = V.uns["spatial"][lib]["scalefactors"]["tissue_hires_scalef"]
xy  = V.obsm["spatial"] * sf                                 # spot centres in hires-image pixels

ct = adata_ref.obs["Subset"].astype(str).values

cmaps={"GC B":"Reds","naive/mem B":"Blues","plasma":"Purples","T":"Greens","NK/ILC":"GnBu","myeloid":"Oranges","stromal":"Greys"}
order=[c for v in lineage.values() for c in v]; assert set(order)==set(ct), set(ct)^set(order)
col={}
for k,v in lineage.items():
    cm=plt.get_cmap(cmaps[k]); 
    for j,c in enumerate(v): col[c]=cm(0.45+0.45*(j/(max(len(v)-1,1))))

focus=["B_GC_LZ","B_GC_DZ","B_Cycling","B_preGC","B_naive","B_mem","B_plasma","T_CD4+_TfH_GC","T_TIM3+","FDC","Macrophages_M1","Monocytes"]


In [ ]:
np.unique(ct)

In [ ]:
# found = genes present in the reference; X = raw counts; E = log1p(CP10k)   (cells x genes)
found, X, E = panel_lognorm(adata_ref, genes_plot)

pct  = pd.DataFrame(X > 0, columns=found).groupby(ct).mean() * 100   # % cells with >0 counts
mean = pd.DataFrame(E,     columns=found).groupby(ct).mean()         # mean log1p(CP10k)
n_ct = pd.Series(ct).value_counts()                                  # cells per type

# row order for the dot plot: cell types grouped by lineage
lineage = {
    "GC B":        ["B_GC_DZ","B_GC_LZ","B_GC_prePB","B_Cycling","B_preGC"],
    "naive/mem B": ["B_naive","B_activated","B_IFN","B_mem"],
    "plasma":      ["B_plasma"],
    "T":           ["T_CD4+_naive","T_CD4+","T_CD4+_TfH","T_CD4+_TfH_GC","T_TfR","T_Treg","T_TIM3+",
                    "T_CD8+_naive","T_CD8+_cytotoxic","T_CD8+_CD161+"],
    "NK/ILC":      ["NK","NKT","ILC"],
    "myeloid":     ["Monocytes","Macrophages_M1","Macrophages_M2","DC_cDC1","DC_cDC2","DC_pDC","DC_CCR7+","Mast"],
    "stromal":     ["FDC","Endo","VSMC"],
}
ct_order = [c for v in lineage.values() for c in v]
assert set(ct_order) == set(ct), set(ct) ^ set(ct_order)

P = pct.loc[ct_order, found]
M = mean.loc[ct_order, found]
U=adata_ref.obsm["X_umap"]


In [ ]:
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

%matplotlib inline

# ---------- Figure 1: UMAP by cell type + dot plot ----------
fig=plt.figure(figsize=(14,8))
gs=fig.add_gridspec(1,2,width_ratios=[1.15,1],wspace=0.32)

ax=fig.add_subplot(gs[0]); rng=np.random.default_rng(0); o=rng.permutation(len(U))
ax.scatter(U[o,0],U[o,1],s=0.4,c=[col[c] for c in ct[o]],lw=0,rasterized=True)

for c in focus:
    m=ct==c; x,y=np.median(U[m],0)
    ax.text(x,y,c,fontsize=7,ha="center",va="center",path_effects=[pe.withStroke(linewidth=2.2,foreground="white")])

ax.set_xticks([]); ax.set_yticks([]); [s.set_visible(False) for s in ax.spines.values()]
ax.set_title("Reference UMAP coloured by cell type (Subset); labelled = types the panel genes point to",loc="left",fontsize=8)

h=[Line2D([],[],marker="o",ls="",color=plt.get_cmap(cmaps[k])(0.75),label=k) for k in lineage]
ax.legend(handles=h,loc="lower left",frameon=False,fontsize=7,markerscale=1.2)
ax.text(0.0,-0.02,"UMAP 1 →",transform=ax.transAxes,fontsize=6,va="top"); ax.text(-0.02,0.0,"UMAP 2 →",transform=ax.transAxes,fontsize=6,rotation=90,ha="right")

ax2=fig.add_subplot(gs[1])
P=pct.loc[order,found]; M=mean.loc[order,found]
yy,xx=np.meshgrid(np.arange(len(order)),np.arange(len(found)),indexing="ij")
sca=ax2.scatter(xx.ravel(),yy.ravel(),s=(P.values.ravel()/100)*60,c=M.values.ravel(),cmap="magma_r",vmin=0,vmax=np.nanmax(M.values),lw=0)
ax2.set_xticks(range(len(found)))
ax2.set_xticklabels([("MS4A1 (CD20)" if g=="MS4A1" else g) for g in found],rotation=90,style="italic")
ax2.set_yticks(range(len(order)))

n_ct = pd.Series(ct).value_counts()          # cells per cell type (skip if already defined)
ax2.set_yticklabels([f"{c} (n={n_ct[c]:,})" for c in order], fontsize=6)

for t,c in zip(ax2.get_yticklabels(),order): t.set_color(col[c] if c in focus else "#555555"); t.set_fontweight("bold" if c in focus else "normal")
ax2.invert_yaxis(); ax2.set_xlim(-0.6,len(found)-0.4); ax2.set_ylim(len(order)-0.4,-0.6)
ax2.set_title("Panel genes by reference cell type",loc="left",fontsize=8)
cb=fig.colorbar(sca,ax=ax2,fraction=0.03,pad=0.02); cb.set_label("mean log1p(CP10k)",fontsize=7)
for p in (25,50,100): ax2.scatter([],[],s=p/100*60,c="#555555",label=f"{p}%")
ax2.legend(title="% cells >0",loc="upper left",bbox_to_anchor=(1.18,0.35),frameon=False,fontsize=6,title_fontsize=7)

plt.show()

filefig = root_figure / "celltype_umap_dotplot.png"
fig.savefig(filefig, dpi=300,bbox_inches="tight"); plt.close(fig)


In [ ]:

# ---------- Figure 2: spatial maps ----------
img=V.uns["spatial"][lib]["images"]["hires"]; sf=V.uns["spatial"][lib]["scalefactors"]["tissue_hires_scalef"]
xy=V.obsm["spatial"]*sf
fig,axes=plt.subplots(3,4,figsize=(12,9.6),gridspec_kw=dict(wspace=0.08,hspace=0.18))
x0,x1=xy[:,0].min()-40,xy[:,0].max()+40; y0,y1=xy[:,1].min()-40,xy[:,1].max()+40
for ax,c in zip(axes.ravel(),focus):
    v=Q[c].values; vmax=np.percentile(v,99.2)
    ax.imshow(img,alpha=0.55)
    o=np.argsort(v)
    s=ax.scatter(xy[o,0],xy[o,1],c=np.clip(v[o],0,vmax),cmap="magma",vmin=0,vmax=vmax,s=5.5,lw=0,alpha=0.9)
    ax.set_xlim(x0,x1); ax.set_ylim(y1,y0); ax.set_xticks([]); ax.set_yticks([])
    [sp_.set_visible(False) for sp_ in ax.spines.values()]
    ax.set_title(c,fontsize=8,color=col[c],fontweight="bold")
    cbar=fig.colorbar(s,ax=ax,fraction=0.04,pad=0.01); cbar.set_ticks([0,round(vmax,1)]); cbar.ax.tick_params(labelsize=6,length=2)
fig.text(0.5,0.075,"cell2location q05 abundance (cells per spot, 'at least this many present'), clipped at the 99.2th percentile per cell type; V1_Human_Lymph_Node, 4,035 spots.",ha="center",fontsize=7)

fig.show()
fig.savefig("celltype_spatial_maps.png",dpi=200,bbox_inches="tight")
plt.close(fig)

# ---------- Figure 3: gene (Visium) vs cell-type abundance correlation ----------
symV=V.var["symbol"].astype(str)
vf,vids=L.resolve_gene_ids(V,found)
Xv=V[:,vids].X; Xv=Xv.toarray() if sps.issparse(Xv) else np.asarray(Xv)
tot=np.asarray(V.X.sum(1)).ravel()
Ev=np.log1p(Xv/tot[:,None]*1e4)
rho=pd.DataFrame([[spearmanr(Ev[:,i],Q[c].values)[0] for c in focus] for i in range(len(vf))],index=vf,columns=focus)

fig,ax=plt.subplots(figsize=(12, 8))
im=ax.imshow(rho.values,cmap="RdBu_r",vmin=-0.8,vmax=0.8,aspect="auto")
for i in range(rho.shape[0]):
    for j in range(rho.shape[1]):
        r=rho.values[i,j]; ax.text(j,i,f"{r:.2f}",ha="center",va="center",fontsize=5.5,color="white" if abs(r)>0.5 else "black")
ax.set_xticks(range(len(focus))); ax.set_xticklabels(focus,rotation=60,ha="right")
for t,c in zip(ax.get_xticklabels(),focus): t.set_color(col[c])
ax.set_yticks(range(len(vf))); ax.set_yticklabels([("MS4A1 (CD20)" if g=="MS4A1" else g) for g in vf],style="italic")
cb=fig.colorbar(im,ax=ax,fraction=0.04,pad=0.02); cb.set_label("Spearman ρ across spots (0 = no association)",fontsize=7); cb.set_ticks([-0.8,-0.4,0,0.4,0.8])
ax.set_title("Visium gene expression vs mapped cell-type abundance (n = 4,035 spots)",loc="left",fontsize=8)
plt.show()

fig.savefig("gene_celltype_spatial_correlation.png",dpi=200,bbox_inches="tight"); plt.close(fig)
rho.round(2).to_csv("gene_celltype_spatial_correlation.csv")
print(rho.idxmax(1).to_dict()); print(rho.max(1).round(2).to_dict())



In [ ]:
"""Per tissue x age-group summary of the B-cell gene panel in the lymphoid-organ scRNA reference.
n_cells, % cells with >0 counts, mean log1p(CP10k). Functions in libs/cell2location_lib.py."""

genes_plot = ["CD19", "MS4A1", "CD79A", "CD79B", "PAX5", "CD22", "CD52",
              "CD53", "CDS2", "IGHD", "IGHM", "CD27", "CD28",
              "LEF1", "CCR7", "SELL", "TCF7", "AICDA"]

dfg = create_gene_age_tissue_groups(adata_ref, genes_plot)   # also writes adata_ref.obs['age_group']
# dfg.to_csv(root0 / 'results' / 'bcell_genes_by_tissue_age.csv', index=False)

print(dfg.shape)
dfg

`CD20` is not a gene symbol in the reference (the gene is `MS4A1`), so the panel uses `MS4A1`, labelled "MS4A1 (CD20)".

Grid below: genes in rows, tissue × age groups in columns; colour = log1p(CP10k) clipped at the 99.2th percentile per gene, one scale per gene row (shared across columns).

**Caveat:** tissue and age are confounded (Tonsil = babies/young only; Spleen/iLN/tLN = adult, one donor; mLN = adult / old adult / elder), so column differences are not pure age effects.

In [ ]:
found, ids = resolve_gene_ids(adata_ref, genes_plot)   # works with symbol or Ensembl var_names
print(len(found), 'of', len(genes_plot), 'genes found')
";".join(found)


In [ ]:
apply_figure_style(sizes=(8, 7, 6))

fig = plot_panel_umap_by_tissue_age(
    adata_ref, genes_plot,
    out_png = root_figure / 'bcell_genes_umap_by_tissue_age.png',
    labels={'MS4A1': 'MS4A1 (CD20)'},
    q=99.2, cmap_num=2
)


<div class="alert alert-warning">
<b>Warning</b>
    
Here we rename genes to ENSEMBL ID for correct matching between single cell and spatial data.
</div>

In [ ]:
len(adata_ref.var.columns)

In [ ]:
adata_ref.var.columns[:50]

In [ ]:
adata_ref.var.columns[50:]

In [ ]:
adata_ref.var.head(6).T.head(12)

In [ ]:
adata_ref.var['symbol']= adata_ref.var.index
# rename 'GeneID-2' as necessary for your data
adata_ref.var.set_index('GeneID-2', drop=True, inplace=True)

# delete unnecessary raw slot (to be removed in a future version of the tutorial)
del adata_ref.raw

In [ ]:
cols = ['symbol', 'GeneName-2', 'feature_types', 'feature_types-0', 'feature_types-1',
       'gene_ids-1', 'gene_ids-4861STDY7135913-0',
       'genome-1', 'n_cells', 'nonz_mean', 'mean_cov_effect_Subset_B_Cycling',
       'mean_cov_effect_Subset_B_GC_DZ', 
       'mean_sample_effectSample_BCP009_Total',
       'mean_sample_effectSample_Human_colon_16S7255677',
       'mean_sample_effectSample_Human_colon_16S7255678',
       'mean_sample_effectSample_Human_colon_16S8000484',
       'mean_sample_effectSample_Pan_T7935494',]

adata_ref.var[cols].head(3).T

<div class="alert alert-info">
<b>Note!</b>
    
Before we estimate the reference cell type signature we recommend to perform very permissive genes selection. We prefer this to standard highly-variable-gene selection because our procedure keeps markers of rare genes while removing most of the uninformative genes.
</div>

The default parameters **cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12** are a good starting point, however, you can increase the cut-off to exclude more genes. 

To preserve marker genes of rare cell types we recommend low **cell_count_cutoff=5**, however, **cell_percentage_cutoff2** and **nonz_mean_cutoff** can be increased to select between 8k-16k genes.

In this 2D histogram, orange rectangle highlights genes excluded based on the combination of number of cells expressing that gene (Y-axis) and average RNA count for cells where the gene was detected (X-axis).

In this case, the downloaded dataset was already filtered using this method, hence no density under the orange rectangle (to be changed in the future version of the tutorial).

In [ ]:
from cell2location.utils.filtering import filter_genes

selected = filter_genes(adata_ref, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)

# filter the object
adata_ref = adata_ref[:, selected].copy()

len(selected), len(adata_ref), selected[:10]

### Estimation of reference cell type signatures (NB regression)

The signatures are estimated from scRNA-seq data, accounting for batch effect, using a Negative binomial regression model.

<div class="alert alert-block alert-message">
<b>Preparing anndata.</b>
    
First, prepare anndata object for the regression model:
</div>

In [ ]:
adata_ref.obs.head(3)

In [ ]:
adata_ref.obsm

In [ ]:
adata_ref.var.head(3)

In [ ]:
adata_ref.var.shape

In [ ]:
adata_ref

### prepare anndata for the regression model

In [ ]:
cell2location.models.RegressionModel.setup_anndata(adata=adata_ref,
                        # 10X reaction / sample / batch
                        batch_key='Sample',
                        # cell type, covariate used for constructing signatures
                        labels_key='Subset',
                        # multiplicative technical effects (platform, 3' vs 5', donor effect)
                        categorical_covariate_keys=['Method']
                       )

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training model.</b>
    
Now we train the model to estimate the reference cell type signatures.
    
Note that to achieve convergence on your data (=to get stabilization of the loss) you may need to increase `max_epochs=250` (See below).
    
Also note that here we are using `batch_size=2500` which is much larger than scvi-tools default and perform training on all cells in the data (`train_size=1`) - both parameters are defaults.
</div>

In [ ]:
adata_file = Path(f"{ref_run_name}/sc.h5ad")
adata_file.exists(), adata_file

In [ ]:
adata_file.exists()

In [ ]:
if not adata_file.exists():
    mod.train(max_epochs=250, accelerator='gpu')

    # In this section, we export the estimated cell abundance (summary of the posterior distribution).
    adata_ref = mod.export_posterior(
        adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500, 'accelerator': 'gpu'}
    )    

    # Save model
    mod.save(f"{ref_run_name}", overwrite=True, save_anndata=True)
    print(f"Model saved with anndata to {ref_run_name}")

    # Save anndata object with results
    adata_ref.write(adata_file)
    print(f"Model saved to {adata_file}")
else:
    # Load AnnData with posterior results (already registered via setup_anndata)
    # adata_ref = sc.read_h5ad(adata_file)

    # Reload the trained model bound to that AnnData
    mod = cell2location.models.RegressionModel.load(f"{ref_run_name}", adata_ref)
    adata_ref = mod.adata

    print(f"Model loaded from {ref_run_name}")

In [ ]:
adata_ref.var.head(3).T

In [ ]:
import torch

saved = torch.load(f"{ref_run_name}/model.pt", map_location="cpu", weights_only=False)
model_genes = pd.Index(saved["var_names"])
adata_genes = adata_ref.var_names

print(len(model_genes), len(adata_genes))
print("missing from adata:", model_genes.difference(adata_genes)[:20].tolist())
print("extra in adata:", adata_genes.difference(model_genes)[:20].tolist())

<div class="alert alert-block alert-message">
<b>Determine if the model needs more training.</b>
</div>

Here, we plot ELBO loss history during training, removing first 20 epochs from the plot.
This plot should have a decreasing trend and level off by the end of training. If it is still decreasing, increase `max_epochs`.

In [ ]:
mod.plot_history(20)

You can compute the 5%, 50% and 95% quantiles of the posterior distribution directly rather than using 1000 samples from the distribution (or any other quantiles). This speeds up application on large datasets and requires less memory - however, posterior mean and standard deviation cannot be computed this way.

In [ ]:
adata_ref = mod.export_posterior(
    adata_ref, use_quantiles=True,
    # choose quantiles
    add_to_varm=["q05","q50", "q95", "q0001"],
    sample_kwargs={'batch_size': 2500, 'accelerator': 'gpu'}
)

In [ ]:
# picks 'means', else 'q50', else 'q05' (export_posterior(use_quantiles=True) writes only quantiles)
inf_aver, summary = get_reference_signatures(adata_ref)
print(f"Using {summary} signatures")

inf_aver.head(6).T.head(8)


In [ ]:
factor_names = adata_ref.uns['mod']['factor_names']
inf_aver = adata_ref.varm['q50_per_cluster_mu_fg'][
    [f'q50_per_cluster_mu_fg_{i}' for i in factor_names]
].copy()
inf_aver.columns = factor_names

inf_aver

<div class="alert alert-block alert-message">
<b>Examine QC plots.</b>
</div>

1. Reconstruction accuracy to assess if there are any issues with inference. This 2D histogram plot should have most observations along a noisy diagonal.

2. The estimated expression signatures are distinct from mean expression in each cluster because of batch effects. For scRNA-seq datasets which do not suffer from batch effect (this dataset does), cluster average expression can be used instead of estimating signatures with a model. When this plot is very different from a diagonal plot (e.g. very low values on Y-axis, density everywhere) it indicates problems with signature estimation.

In [ ]:
mod.plot_QC(summary_name="q50")    # median signatures

<div class="alert alert-block alert-message">
<b>Extracting reference cell types signatures as a pd.DataFrame.</b>
    
All parameters of the a Negative Binomial regression model are exported into reference anndata object, however for spatial mapping we just need the estimated expression of every gene in every cell type. Here we extract that from standard output:
</div>

In [ ]:
[x for x in adata_ref.var.columns if 'cluster' in x]

In [ ]:
# export estimated expression in each cluster
# (the original hard-coded 'means_per_cluster_mu_fg' -> KeyError after a quantile-only export)
inf_aver, summary = get_reference_signatures(adata_ref, prefer=("means", "q50"))
print(f"Using {summary} signatures")
inf_aver.iloc[0:5, 0:5]


### Cell2location: spatial mapping

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

len(intersect), len(adata_vis.var_names), len(inf_aver.index)

#### prepare anndata for cell2location model

In [ ]:
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")

<div class="alert alert-info">
<b> Important </b>
    
To use cell2location spatial mapping model, you need to specify 2 user-provided hyperparameters (**N_cells_per_location** and **detection_alpha**) 

For detailed guidance on setting these hyperparameters and their impact see [the flow diagram and the note](https://github.com/BayraktarLab/cell2location/blob/master/docs/images/Note_on_selecting_hyperparameters.pdf).
</div>

**Choosing hyperparameter `N_cells_per_location`!**

It is useful to adapt the expected cell abundance `N_cells_per_location` to every tissue</b>. This value can be estimated from paired histology images and as described in the note above. Change the value presented in this tutorial (`N_cells_per_location=30`) to the value observed in your your tissue.

**Choosing hyperparameter `detection_alpha`!**

To improve accuracy & sensitivity on datasets with large technical variability in RNA detection sensitivity within the slide/batch - you need to relax regularisation of per-location normalisation (use `detection_alpha=20`). High technical variability in RNA detection sensitivity is present in your sample when you observe the spatial distribution of total RNA count per location that doesn't match expected cell numbers based on histological examination.

We initially opted for high regularisation (`detection_alpha=200`) as a default because the mouse brain & human lymph node datasets used in our paper have low technical effects and using high regularisation strenght improves consistencly between total estimated cell abundance per location and the nuclei count quantified from histology ([Fig S8F in cell2location paper](https://static-content.springer.com/esm/art%3A10.1038%2Fs41587-021-01139-4/MediaObjects/41587_2021_1139_MOESM1_ESM.pdf)). However, in many collaborations, we see that Visium experiments on human tissues suffer from technical effects. This motivates the new default value of `detection_alpha=20` and the recommendation of testing both settings on your data (`detection_alpha=20` and `detection_alpha=200`).

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_vis, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
)
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training cell2location:</b>
</div>

In [ ]:
adata_file_run = run_name / "sp.h5ad"
adata_file_run.exists(), adata_file_run


In [ ]:
if not adata_file_run.exists():
    mod.train(max_epochs=15000,
            # train using full data (batch_size=None)
            batch_size=None,
            # use all data points in training because
            # we need to estimate cell abundance at all locations
            train_size=1,
            accelerator='gpu',
            )

    # In this section, we export the estimated cell abundance (summary of the posterior distribution).
    adata_vis = mod.export_posterior(
        adata_vis, use_quantiles=True,
        add_to_obsm=["q05", "q50", "q95", "q0001"],
        sample_kwargs={'batch_size': mod.adata.n_obs, 'accelerator': 'gpu'} 
        # sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs, 'accelerator': 'gpu'}
    )

    # Compute expected expression per cell type
    expected_dict = mod.module.model.compute_expected_per_cell_type(
        mod.samples["post_sample_q05"], mod.adata_manager
    )

    # Add to anndata layers
    for i, n in enumerate(mod.factor_names_):
        adata_vis.layers[n] = expected_dict['mu'][i]

    # Save model
    mod.save(f"{run_name}", overwrite=True)
    print(f"Model saved to {run_name}")

    # Save anndata object with results
    adata_vis.write(adata_file_run)
    print(f"adata_vis saved to {adata_file_run}")
else:
    # Load AnnData with posterior abundances in .obsm (already registered via setup_anndata)
    adata_vis = sc.read_h5ad(adata_file_run)
    print(f"adata_vis read from {adata_file_run}")

    # Reload the trained spatial model bound to that AnnData
    mod = cell2location.models.Cell2location.load(f"{run_name}", adata_vis)
    print(f"Model loaded from {run_name}")

    # mod.samples is not saved with the model: rebuild it (needed by plot_QC and compute_expected_per_cell_type)
    adata_vis = ensure_posterior_samples(mod, adata_vis, quantiles=["q05", "q50", "q95", "q0001"], accelerator="gpu")
    print("posterior samples:", sorted(mod.samples.keys()))

# Common to both branches: copy 5% quantile abundances into .obs for plotting
factor_names = adata_vis.uns['mod']['factor_names']
adata_vis.obs[factor_names] = adata_vis.obsm['q05_cell_abundance_w_sf']


In [ ]:
mod.adata.n_obs

### plot ELBO loss history during training, removing first 100 epochs from the plot

In [ ]:
mod.plot_history(1000)
plt.legend(labels=['full data training']);

<div class="alert alert-block alert-message">
<b>Exporting estimated posterior distributions of cell abundance and saving results:</b>
</div>

The model and output h5ad can be loaded later like this:
```python
adata_file = f"{run_name}/sp.h5ad"
adata_vis = sc.read_h5ad(adata_file)
mod = cell2location.models.Cell2location.load(f"{run_name}", adata_vis)
```

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [ ]:
mod.plot_QC(summary_name="q50")    # median signatures

When intergrating multiple spatial batches and when working with datasets that have substantial variation of detected RNA within slides (that cannot be explained by high cellular density in the histology), it is important to assess whether cell2location normalised those effects. You expect to see similar total cell abundance across batches but distinct RNA detection sensitivity (both estimated by cell2location). You expect total cell abundance to mirror high cellular density in the histology.

```python
fig = mod.plot_spatial_QC_across_batches()
```

## Visualising cell abundance in spatial coordinates <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_vis.obs[adata_vis.uns['mod']['factor_names']]= adata_vis.obsm['q05_cell_abundance_w_sf']

# select one slide
from cell2location.utils import select_slide
slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')

# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):

    sc.pl.spatial(slide, cmap='RdBu_r',
                  # show first 8 cell types
                  color=['B_Cycling', 'B_GC_LZ', 'T_CD4+_TfH_GC', 'FDC',
                         'B_naive', 'T_CD4+_naive', 'B_plasma', 'Endo'],
                  ncols=4, size=1.3,
                  img_key='hires',
                  # limit color scale at 99.2% quantile of cell abundance
                  vmin=0, vmax='p99.2'
                 )

In [ ]:
# Now we use cell2location plotter that allows showing multiple cell types in one panel
from cell2location.plt import plot_spatial

# select up to 6 clusters
# Follicular dendritic cells (FDCs)
clust_labels = ['T_CD4+_naive', 'B_naive', 'FDC']
clust_col = [str(i) for i in clust_labels]# in case column names differ from labels

slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')

with mpl.rc_context({'figure.figsize': (15, 15)}):
    fig = plot_spatial(
        adata=slide,
        # labels to show on a plot
        color=clust_col, labels=clust_labels,
        show_img=True,
        # 'fast' (white background) or 'dark_background'
        style='fast',
        # limit color scale at 99.2% quantile of cell abundance
        max_color_quantile=0.992,
        # size of locations (adjust depending on figure size)
        circle_diameter=6,
        colorbar_position='right'
    )

### Downstream analysis <a class="anchor" id="Downstream-analysis"></a>

#### Identifying discrete tissue regions by Leiden clustering<a class="anchor" 

id="Identifying-discrete-tissue-regions-by-Leiden-clustering"></a>

We identify tissue regions that differ in their cell composition by clustering locations using cell abundance estimated by cell2location.

We find tissue regions by clustering Visium spots using estimated cell abundance each cell type. We constuct a K-nearest neigbour (KNN) graph representing similarity of locations in estimated cell abundance and then apply Leiden clustering. The number of KNN neighbours should be adapted to size of dataset and the size of anatomically defined regions (e.i. hippocampus regions are rather small compared to size of the brain so could be masked by large `n_neighbors`). This can be done for a range KNN neighbours and Leiden clustering resolutions until a clustering matching the anatomical structure of the tissue is obtained.

The clustering is done jointly across all Visium sections / batches, hence the region identities are directly comparable. When there are strong technical effects between multiple batches (not the case here) `sc.external.pp.bbknn` can be in principle used to account for those effects during the KNN construction.

The resulting clusters are saved in `adata_vis.obs['region_cluster']`.

In [ ]:
# compute KNN using the cell2location output stored in adata.obsm
sc.pp.neighbors(adata_vis, use_rep='q05_cell_abundance_w_sf',
                n_neighbors = 15)

# Cluster spots into regions using scanpy
sc.tl.leiden(adata_vis, resolution=1.1)

# add region as categorical variable
adata_vis.obs["region_cluster"]= adata_vis.obs["leiden"].astype("category")

We can use the location composition similarity graph to build a joint integrated UMAP representation of all section/Visium batches.

In [ ]:
# compute UMAP using KNN graph based on the cell2location output
sc.tl.umap(adata_vis, min_dist = 0.3, spread = 1)

# show regions in UMAP coordinates
with mpl.rc_context({'axes.facecolor':  'white',
                     'figure.figsize': [8, 8]}):
    sc.pl.umap(adata_vis, color=['region_cluster'], size=30,
               color_map = 'RdPu', ncols = 2, legend_loc='on data',
               legend_fontsize=20)
    sc.pl.umap(adata_vis, color=['sample'], size=30,
               color_map = 'RdPu', ncols = 2,
               legend_fontsize=20)

# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):
    sc.pl.spatial(adata_vis, color=['region_cluster'],
                  size=1.3, img_key='hires', alpha=0.5)

### Identifying cellular compartments / tissue zones using matrix factorisation (NMF) <a name="Identifying-cellular-compartments-/-tissue-zones-using-matrix-factorisation-(NMF)"></a>

Here, we use the cell2location mapping results to identify the spatial co-occurrence of cell types in order to better understand the tissue organisation and predict cellular interactions. We performed non-negative matrix factorization (NMF) of the cell type abundance estimates from cell2location ([paper section 4, Fig 4D](https://www.nature.com/articles/s41587-021-01139-4)). Similar to the established benefits of applying NMF to conventional scRNA-seq, the additive NMF decomposition yielded a grouping of spatial cell type abundance profiles into components that capture co-localised cell types ([Supplemenary Methods section 4.2, p. 60](https://www.nature.com/articles/s41587-021-01139-4#Sec50)). This NMF-based decomposition naturally accounts for the fact that multiple cell types and microenvironments can co-exist at the same Visium locations (see [paper Fig S20, p. 34](https://www.nature.com/articles/s41587-021-01139-4#Sec50)), while sharing information across tissue areas (e.g. individual germinal centres).


<div class="alert alert-block alert-primary">
<b>Tip</b>
    
In practice, it is better to train NMF for a range of factors $R={5, .., 30}$ and select $R$ as a balance between capturing fine-grained and splitting known well-established tissue zones.

If you want to find a few most disctinct cellular compartments, use a small number of factors.   
If you want to find very strong co-location signal and assume that most cell types don't co-locate, use a lot of factors (> 30 - used here).
</div>

Below we show how to perform this analysis. To aid this analysis, we wrapped the analysis shown the notebook on advanced downstream analysis into a pipeline that automates training of the NMF model with varying number of factors:

In [ ]:
from cell2location import run_colocation

res_dict, adata_vis = run_colocation(
    adata_vis,
    model_name='CoLocatedGroupsSklearnNMF',
    train_args={
      'n_fact': np.arange(11, 13), # IMPORTANT: use a wider range of the number of factors (5-30)
      'sample_name_col': 'sample', # columns in adata_vis.obs that identifies sample
      'n_restarts': 3 # number of training restarts
    },
    # the hyperparameters of NMF can be also adjusted:
    model_kwargs={'alpha': 0.01, 'init': 'random', "nmf_kwd_args": {"tol": 0.000001}},
    export_args={'path': f'{run_name}/CoLocatedComb/'}
)

For every factor number, the model produces the following list of folder outputs:

`cell_type_fractions_heatmap/`: a dot plot of the estimated NMF weights of cell types (rows) across NMF components (columns)

`cell_type_fractions_mean/`: the data used for dot plot

`factor_markers/`: tables listing top 10 cell types most speficic to each NMF factor

`models/`: saved NMF models

`predictive_accuracy/`: 2D histogram plot showing how well NMF explains cell2location output

`spatial/`: NMF weights across locatinos in spatial coordinates

`location_factors_mean/`: the data used for the plot in spatial coordiantes

`stability_plots/`: stability of NMF weights between training restarts

Key output that you want to examine are the files in `cell_type_fractions_heatmap/` which show a dot plot of the estimated NMF weights of cell types (rows) across NMF components (columns) which correspond to cellular compartments. Shown are relative weights, normalized across components for every cell type.

<div class="alert alert-block alert-primary">
<b>Tip</b>
    
The NMF model output such as factor loadings are stored in `adata.uns[f"mod_coloc_n_fact{n_fact}"]` in a similar output format as main cell2location results in `adata.uns['mod']`.
</div>

In [ ]:
# Here we plot the NMF weights (Same as saved to `cell_type_fractions_heatmap`)
res_dict['n_fact12']['mod'].plot_cell_type_loadings()

### Estimate cell-type specific expression of every gene in the spatial data (needed for NCEM) <a name="Estimate-cell-type-specific-expression-of-every-gene-in-the-spatial-data-(needed-for-NCEM)"></a>

The cell-type specific expression of every gene at every spatial location in the spatial data enables learning cell communication with NCEM model using Visium data (https://github.com/theislab/ncem).

To derive this, we adapt the approach of estimating conditional expected expression proposed by [RCTD (Cable et al)](https://pubmed.ncbi.nlm.nih.gov/33603203/) method.

With cell2location, we can look at the posterior distribution rather than just point estimates of cell type specific expression (see `mod.samples.keys()` and next section on using full distribution).

Note that this analysis requires substantial amount of RAM memory and thefore doesn't work on free Google Colab (12 GB limit).

In [ ]:
mod

In [ ]:
print(list(mod.samples.keys()))

In [ ]:
# Compute expected expression per cell type
expected_dict = mod.module.model.compute_expected_per_cell_type(
    mod.samples["post_sample_q05"], mod.adata_manager
)

# Add to anndata layers
for i, n in enumerate(mod.factor_names_):
    adata_vis.layers[n]= expected_dict['mu'][i]

# Save anndata object with results (adata_file_run, not adata_file, which is the reference sc.h5ad path)
adata_vis.write(adata_file_run)
adata_file_run

<div class="alert alert-block alert-primary">
<b>Plotting cell-type specific expression of genes in spatial coordinates.</b>
    
Below we plot the cell-type specific expression of genes (rows, second to last columns) compared to total expression of those genes (first column).

Here we highlight *CD3D*, pan T-cell marker expressed by 2 subtypes of T cells in distinct locations but not expressed by co-located B cells, that instead express *CR2* gene.
</div>

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
# list cell types and genes for plotting
ctypes = ['T_CD4+_TfH_GC', 'T_CD4+_naive', 'B_GC_LZ']
genes = ['CD3D', 'CR2']

with mpl.rc_context({'axes.facecolor':  'black'}):
    # select one slide
    slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')

    # from tutorial_utils import plot_genes_per_cell_type
    c2l.plot_genes_per_cell_type(slide, genes, ctypes);

In [ ]:
slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')

In [ ]:
slide.var

In [ ]:
slide.layers

Note that `plot_genes_per_cell_type` function often need customization so it is not included into cell2location package - you need to copy it from https://github.com/BayraktarLab/cell2location/blob/master/docs/notebooks/tutorial_utils.py to use on your system.

## Advanced use <a name="advanced"></a>

### Working with the posterior distribution and computing arbitrary quantiles <a name="Working-with-the-posterior-distribution-and-computing-arbitrary-quantiles"></a>

In addition to the posterior distribution mean, std and quantiles presented earlier in the notebook you can fetch an arbitrary number of samples from the posterior distribution. To limit memory use, it could be beneficial to select particular varibles in the model.

Note that this analysis requires substantial amount RAM memory and thefore doesn't work on Google Colab.

In [ ]:
# Get posterior distribution samples for specific variables
samples_w_sf = mod.sample_posterior(num_samples=1000, accelerator='gpu', return_samples=True,
                                    batch_size=2020,
                                    return_sites=['w_sf', 'm_g', 'u_sf_mRNA_factors'])
# samples_w_sf['posterior_samples']contains 1000 samples as arrays with dim=(num_samples, ...)
samples_w_sf['posterior_samples']['w_sf'].shape

Finally, it could be useful to compute arbitrary quantiles of the posterior distribution.

In [ ]:
# Compute any quantile of the posterior distribution
medians = mod.posterior_quantile(q=0.5, batch_size=mod.adata.n_obs, accelerator='gpu')

with mpl.rc_context({'axes.facecolor':  'white',
                     'figure.figsize': [5, 5]}):
    if 'post_sample_means' in mod.samples:
        means_w_sf = mod.samples['post_sample_means']['w_sf']
    else:   # quantile-only export: use the posterior samples drawn in the cell above
        means_w_sf = samples_w_sf['posterior_samples']['w_sf'].mean(0)
    plt.scatter(medians['w_sf'].flatten(), np.asarray(means_w_sf).flatten());
    plt.xlabel('median');
    plt.ylabel('mean');

### Modules and their versions used for this analysis


Useful for debugging and reporting issues.

In [ ]:
cell2location.utils.list_imported_modules()

In [ ]:
cell2location.__version__